In [1]:
import pandas as pd
import numpy as np
import re
import math
import tldextract
from urllib.parse import urlparse
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported!")

✅ Libraries imported!


In [2]:
df = pd.read_csv(r'D:\malicious-url-detection\data\malicious_phish.csv')
print(f"Shape: {df.shape}")
df.head()

Shape: (651191, 2)


,url,type
0,br-icloud.com.br,phishing
1,mp3raid.com/music/krizz_kaliko.html,benign
2,bopsecrets.org/rexroth/cr/1.htm,benign
3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,http://adventure-nicaragua.net/index.php?optio...,defacement


In [3]:
def get_basic_features(url):
    return {
        'url_length'        : len(url),
        'num_dots'          : url.count('.'),
        'num_hyphens'       : url.count('-'),
        'num_underscores'   : url.count('_'),
        'num_slashes'       : url.count('/'),
        'num_at'            : url.count('@'),
        'num_question'      : url.count('?'),
        'num_equals'        : url.count('='),
        'num_ampersand'     : url.count('&'),
        'num_digits'        : sum(c.isdigit() for c in url),
        'num_special_chars' : len(re.findall(r'[^a-zA-Z0-9]', url)),
    }

# Test on one URL
sample = df['url'][0]
print(f"URL: {sample}")
print(get_basic_features(sample))

URL: br-icloud.com.br
{'url_length': 16, 'num_dots': 2, 'num_hyphens': 1, 'num_underscores': 0, 'num_slashes': 0, 'num_at': 0, 'num_question': 0, 'num_equals': 0, 'num_ampersand': 0, 'num_digits': 0, 'num_special_chars': 3}


In [4]:
def get_security_features(url):
    return {
        'has_https'         : 1 if url.startswith('https') else 0,
        'has_http'          : 1 if url.startswith('http') else 0,
        'has_ip_address'    : 1 if re.search(r'\d+\.\d+\.\d+\.\d+', url) else 0,
        'has_at_symbol'     : 1 if '@' in url else 0,
        'has_double_slash'  : 1 if '//' in url[7:] else 0,
        'has_prefix_suffix' : 1 if '-' in urlparse(url).netloc else 0,
    }

print(get_security_features(df['url'][0]))

{'has_https': 0, 'has_http': 0, 'has_ip_address': 0, 'has_at_symbol': 0, 'has_double_slash': 0, 'has_prefix_suffix': 0}


In [5]:
def get_domain_features(url):
    try:
        extracted = tldextract.extract(url)
        parsed    = urlparse(url)
        domain    = extracted.domain
        subdomain = extracted.subdomain
        suffix    = extracted.suffix
        path      = parsed.path
        query     = parsed.query

        return {
            'domain_length'    : len(domain),
            'subdomain_length' : len(subdomain),
            'tld_length'       : len(suffix),
            'path_length'      : len(path),
            'query_length'     : len(query),
            'num_subdomains'   : len(subdomain.split('.')) if subdomain else 0,
        }
    except:
        return {
            'domain_length'    : 0,
            'subdomain_length' : 0,
            'tld_length'       : 0,
            'path_length'      : 0,
            'query_length'     : 0,
            'num_subdomains'   : 0,
        }

print(get_domain_features(df['url'][0]))

{'domain_length': 9, 'subdomain_length': 0, 'tld_length': 6, 'path_length': 16, 'query_length': 0, 'num_subdomains': 0}


In [6]:
def get_entropy(url):
    prob = [url.count(c) / len(url) for c in set(url)]
    return -sum(p * math.log2(p) for p in prob)

def get_entropy_features(url):
    return {
        'url_entropy' : round(get_entropy(url), 4)
    }

print(get_entropy_features(df['url'][0]))

{'url_entropy': 3.375}


In [7]:
def extract_all_features(url):
    features = {}
    features.update(get_basic_features(url))
    features.update(get_security_features(url))
    features.update(get_domain_features(url))
    features.update(get_entropy_features(url))
    return features

# Test on one URL
print(extract_all_features(df['url'][0]))

{'url_length': 16, 'num_dots': 2, 'num_hyphens': 1, 'num_underscores': 0, 'num_slashes': 0, 'num_at': 0, 'num_question': 0, 'num_equals': 0, 'num_ampersand': 0, 'num_digits': 0, 'num_special_chars': 3, 'has_https': 0, 'has_http': 0, 'has_ip_address': 0, 'has_at_symbol': 0, 'has_double_slash': 0, 'has_prefix_suffix': 0, 'domain_length': 9, 'subdomain_length': 0, 'tld_length': 6, 'path_length': 16, 'query_length': 0, 'num_subdomains': 0, 'url_entropy': 3.375}


In [8]:
print("Extracting features from all URLs... please wait ⏳")

feature_df = pd.DataFrame(df['url'].apply(extract_all_features).tolist())
feature_df['type'] = df['type'].values

print(f"✅ Done! Feature matrix shape: {feature_df.shape}")
feature_df.head()

Extracting features from all URLs... please wait ⏳
✅ Done! Feature matrix shape: (651191, 25)


,url_length,num_dots,num_hyphens,num_underscores,num_slashes,num_at,num_question,num_equals,num_ampersand,num_digits,...,has_double_slash,has_prefix_suffix,domain_length,subdomain_length,tld_length,path_length,query_length,num_subdomains,url_entropy,type
0,16,2,1,0,0,0,0,0,0,0,...,0,0,9,0,6,16,0,0,3.3750,phishing
1,35,2,0,1,2,0,0,0,0,1,...,0,0,7,0,3,35,0,0,4.0791,benign
2,31,2,0,0,3,0,0,0,0,1,...,0,0,10,0,3,31,0,0,3.7081,benign
3,88,3,1,2,3,0,1,4,3,7,...,0,1,14,3,2,10,49,1,4.6603,defacement
4,235,2,1,1,3,0,1,3,2,22,...,0,1,19,0,3,10,194,0,5.4913,defacement


In [9]:
feature_df.to_csv(r'D:\malicious-url-detection\data\features.csv', index=False)
print("✅ Features saved to data/features.csv!")

✅ Features saved to data/features.csv!


In [1]:
TRUSTED_DOMAINS = {
    'google', 'youtube', 'facebook', 'twitter', 'instagram',
    'linkedin', 'microsoft', 'apple', 'amazon', 'netflix',
    'github', 'wikipedia', 'reddit', 'yahoo', 'bing', 'adobe',
    'dropbox', 'spotify', 'paypal', 'ebay', 'twitter', 'whatsapp',
    'telegram', 'zoom', 'slack', 'wordpress', 'shopify', 'salesforce'
}

print(f"✅ Trusted domains list ready! ({len(TRUSTED_DOMAINS)} domains)")

✅ Trusted domains list ready! (27 domains)


In [4]:
import pandas as pd
import numpy as np
import re
import math
import tldextract
from urllib.parse import urlparse
import warnings
warnings.filterwarnings('ignore')

TRUSTED_DOMAINS = {
    'google', 'youtube', 'facebook', 'twitter', 'instagram',
    'linkedin', 'microsoft', 'apple', 'amazon', 'netflix',
    'github', 'wikipedia', 'reddit', 'yahoo', 'bing', 'adobe',
    'dropbox', 'spotify', 'paypal', 'ebay', 'twitter', 'whatsapp',
    'telegram', 'zoom', 'slack', 'wordpress', 'shopify', 'salesforce'
}

print("✅ All imports ready!")

✅ All imports ready!


In [5]:
def extract_all_features_v2(url):
    # --- Basic Features ---
    basic = {
        'url_length'        : len(url),
        'num_dots'          : url.count('.'),
        'num_hyphens'       : url.count('-'),
        'num_underscores'   : url.count('_'),
        'num_slashes'       : url.count('/'),
        'num_at'            : url.count('@'),
        'num_question'      : url.count('?'),
        'num_equals'        : url.count('='),
        'num_ampersand'     : url.count('&'),
        'num_digits'        : sum(c.isdigit() for c in url),
        'num_special_chars' : len(re.findall(r'[^a-zA-Z0-9]', url)),
    }

    # --- Security Features ---
    security = {
        'has_https'         : 1 if url.startswith('https') else 0,
        'has_http'          : 1 if url.startswith('http') else 0,
        'has_ip_address'    : 1 if re.search(r'\d+\.\d+\.\d+\.\d+', url) else 0,
        'has_at_symbol'     : 1 if '@' in url else 0,
        'has_double_slash'  : 1 if '//' in url[7:] else 0,
        'has_prefix_suffix' : 1 if '-' in urlparse(url).netloc else 0,
    }

    # --- Domain Features ---
    try:
        extracted  = tldextract.extract(url)
        parsed     = urlparse(url)
        domain     = extracted.domain
        subdomain  = extracted.subdomain
        suffix     = extracted.suffix
        path       = parsed.path
        query      = parsed.query

        domain_f = {
            'domain_length'      : len(domain),
            'subdomain_length'   : len(subdomain),
            'tld_length'         : len(suffix),
            'path_length'        : len(path),
            'query_length'       : len(query),
            'num_subdomains'     : len(subdomain.split('.')) if subdomain else 0,
            'is_trusted_domain'  : 1 if domain.lower() in TRUSTED_DOMAINS else 0,
        }
    except:
        domain_f = {
            'domain_length'      : 0,
            'subdomain_length'   : 0,
            'tld_length'         : 0,
            'path_length'        : 0,
            'query_length'       : 0,
            'num_subdomains'     : 0,
            'is_trusted_domain'  : 0,
        }

    # --- Entropy Feature ---
    prob      = [url.count(c) / len(url) for c in set(url)]
    entropy   = -sum(p * math.log2(p) for p in prob)
    entropy_f = {'url_entropy': round(entropy, 4)}

    # --- Suspicious Keywords Feature ---
    suspicious_keywords = [
        'login', 'verify', 'secure', 'account', 'update',
        'banking', 'confirm', 'password', 'signin', 'wallet',
        'free', 'lucky', 'winner', 'click', 'setup', 'install'
    ]
    keyword_f = {
        'has_suspicious_keyword': 1 if any(
            kw in url.lower() for kw in suspicious_keywords
        ) else 0
    }

    # --- Combine All ---
    features = {}
    features.update(basic)
    features.update(security)
    features.update(domain_f)
    features.update(entropy_f)
    features.update(keyword_f)

    return features

print("✅ Updated feature extractor v2 ready!")

✅ Updated feature extractor v2 ready!


In [6]:
df_raw = pd.read_csv(r'D:\malicious-url-detection\data\malicious_phish.csv')

print("Re-extracting features with improvements... ⏳")
feature_df_v2 = pd.DataFrame(df_raw['url'].apply(extract_all_features_v2).tolist())
feature_df_v2['type'] = df_raw['type'].values

print(f"✅ Done! New feature matrix shape: {feature_df_v2.shape}")
print(f"New features added: {feature_df_v2.shape[1] - 18} extra columns")
feature_df_v2.head()

Re-extracting features with improvements... ⏳
✅ Done! New feature matrix shape: (651191, 27)
New features added: 9 extra columns


,url_length,num_dots,num_hyphens,num_underscores,num_slashes,num_at,num_question,num_equals,num_ampersand,num_digits,...,domain_length,subdomain_length,tld_length,path_length,query_length,num_subdomains,is_trusted_domain,url_entropy,has_suspicious_keyword,type
0,16,2,1,0,0,0,0,0,0,0,...,9,0,6,16,0,0,0,3.3750,0,phishing
1,35,2,0,1,2,0,0,0,0,1,...,7,0,3,35,0,0,0,4.0791,0,benign
2,31,2,0,0,3,0,0,0,0,1,...,10,0,3,31,0,0,0,3.7081,0,benign
3,88,3,1,2,3,0,1,4,3,7,...,14,3,2,10,49,1,0,4.6603,0,defacement
4,235,2,1,1,3,0,1,3,2,22,...,19,0,3,10,194,0,0,5.4913,0,defacement


In [7]:
feature_df_v2.to_csv(
    r'D:\malicious-url-detection\data\features_v2.csv', index=False
)
print("✅ Saved to data/features_v2.csv!")

✅ Saved to data/features_v2.csv!
